# Model 1: the short training walkthrough

**Start here for the oral defense.** Follow five steps: choose settings, load data, create the model,
define one training pass, then train and save.

Model 1 predicts three things: whether there is a target error, its category, and its location.
For an invented example, `Yesterday I polish the lantern.` has a past-tense error at `polish`.
Model 2 handles the explanation and correction later.

The model and training method are unchanged. Supporting code handles token alignment, padding, and
validation metrics. You can read the [full walkthrough](details/model1_training_details.ipynb) later.
This notebook reports training loss only. The full version also reports training classification metrics.
Both versions use the same validation metrics and choose the best model by validation loss.

## 1. Choose the settings
Set the private file paths and training settings in one place. An epoch is one pass through the training data.
The learning rate controls the size of the optimizer's updates. The seed helps make runs repeatable.
The prepared data and trained weights stay outside this public repository.

In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer
from models.model1 import Detector, ModelConfig, ENCODER_REVISION, multitask_loss
from models.checkpoints import model_inputs, save_checkpoint
from models.training import seed_everything, training_loaders, evaluate

In [ ]:
PRIVATE_ROOT = Path.home() / "patternlab-private"
TRAIN_FILE = PRIVATE_ROOT / "model1" / "train.jsonl"
VALIDATION_FILE = PRIVATE_ROOT / "model1" / "validation.jsonl"
OUTPUT = PRIVATE_ROOT / "runs" / "model1-presentation"
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
seed_everything(SEED)

## 2. Load the prepared data
Each sentence has labels that tell the model what it should predict.
The tokenizer splits text into tokens. The supporting loader aligns character labels with those tokens
and groups examples into batches. It also rejects invalid records and overlap between train and validation.
We print counts, not private sentences.

In [ ]:
assert TRAIN_FILE.is_file() and VALIDATION_FILE.is_file(), "Set the private data paths first."
assert not OUTPUT.exists(), "Use a new output folder."
config = ModelConfig(encoder_revision=ENCODER_REVISION, max_length=256)
tokenizer = AutoTokenizer.from_pretrained(config.encoder_name, revision=ENCODER_REVISION, use_fast=True)
train_loader, validation_loader = training_loaders(
    TRAIN_FILE, VALIDATION_FILE, tokenizer, BATCH_SIZE, config.max_length, SEED)
print({"training_examples": len(train_loader.dataset), "validation_examples": len(validation_loader.dataset)})

## 3. Create the model
RoBERTa supplies fixed token representations. The Bi-LSTM reads the sentence in both directions.
Attention combines the token states. Output layers predict error presence, category, and location.
Only the Bi-LSTM, attention, and output layers learn. AdamW is the optimizer that updates their weights.
The full network is in [models/model1.py](models/model1.py).

In [ ]:
model = Detector.from_pretrained(config).to(DEVICE)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)

## 4. Define one training pass
For each batch, the model makes predictions. The loss measures how far they are from the labels.
The combined loss covers detection, category, and location. Category and location use error examples only.

`backward()` calculates gradients, which show how weights affect the loss.
Gradient clipping limits large gradients. `step()` updates the trainable weights.
The function returns the average loss across all examples in the epoch.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    examples = 0
    for batch in loader:
        batch = {name: value.to(DEVICE) for name, value in batch.items()}
        predictions = model(**model_inputs(batch))
        loss = multitask_loss(predictions, batch)["total"]
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        count = len(batch["has_error"])
        total_loss += loss.item() * count
        examples += count
    return total_loss / examples

## 5. Train, validate, and save
After each epoch, validation measures performance without changing the weights.
The lowest validation loss selects `best.pt`. `last.pt` stores the final epoch.
We save the tokenizer and settings so the running notebook and website can load the model.
The test set does not enter this process. A new run still needs held-out evaluation before reporting results.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=False)
tokenizer.save_pretrained(OUTPUT / "tokenizer")
history = []
best_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    validation = evaluate(model, validation_loader, DEVICE)
    history.append({"epoch": epoch, "train": {"loss": train_loss}, "validation": validation,
                    "synthetic_software_check": False})
    print({"epoch": epoch, "train_loss": train_loss, "validation_loss": validation["loss"]})
    save_checkpoint(OUTPUT / "last.pt", model, optimizer, epoch, SEED)
    if validation["loss"] < best_loss:
        best_loss = validation["loss"]
        save_checkpoint(OUTPUT / "best.pt", model, optimizer, epoch, SEED)
(OUTPUT / "history.json").write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")

## What to explain first
1. The input is a sentence with training labels.
2. Model 1 predicts error presence, one category, and one location.
3. RoBERTa stays fixed. The added network layers learn.
4. Loss compares predictions with labels. The optimizer updates weights using gradients.
5. Validation loss selects the saved model.

This is enough for the first walkthrough. Use the detailed notebook when you want to inspect token alignment,
network layers, and metric calculations. Clear all outputs before sharing an executed notebook.